In [ ]:
from textblob  import TextBlob
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamb/go-emotions-google-emotions-dataset")

print("Path to dataset files:", path)

100%|██████████| 8.68M/8.68M [00:01<00:00, 8.14MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shivamb/go-emotions-google-emotions-dataset/versions/1


In [ ]:
df = pd.read_csv('/root/.cache/kagglehub/datasets/shivamb/go-emotions-google-emotions-dataset/versions/1/go_emotions_dataset.csv')
df.head()

,id,text,example_very_unclear,admiration,amusement,anger,annoyance,approval,caring,confusion,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,eew5j0j,That game hurt.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,eemcysk,>sexuality shouldn’t be a grouping category I...,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ed2mah1,"You do right, if you don't care then fuck 'em!",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,eeibobj,Man I love reddit.,False,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,eda6yn6,"[NAME] was nowhere near them, he was by the Fa...",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [ ]:
from nltk.sem.drt import Tokens
def clean_text(text):
  text = text.lower()
  text = re.sub(r'http\s+|www\S+|https\S+','',text,flags=re.MULTILINE)
  text = re.sub(r'\@\w+|\#','',text)
  text = text.translate(str.maketrans('','',string.punctuation))
  text = re.sub(r'\d+','',text)
  text = text.strip()

  stop_words = set(stopwords.words('english'))
  lemmatizer = WordNetLemmatizer()

  Tokens = text.split()
  token = [lemmatizer.lemmatize(word) for word in Tokens if not word in stop_words]
  text = ' '.join(token)

  return text

In [ ]:
import numpy as np

# text vectorization
verctorize = CountVectorizer()
x = verctorize.fit_transform(df['text'])

# Convert TextBlob sentiment to numerical labels for classification
# Polarity > 0: Positive (1), Polarity < 0: Negative (-1), Polarity == 0: Neutral (0)
y_labels = []
for s in df['text'].apply(lambda x: TextBlob(x).sentiment):
    if s.polarity > 0:
        y_labels.append(1)
    elif s.polarity < 0:
        y_labels.append(-1)
    else:
        y_labels.append(0)

y = np.array(y_labels) # Convert to a dense numpy array

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
log = LogisticRegression()
log.fit(x_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
y_pred = log.predict(x_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))

Accuracy: 0.9666706119067345
Confusion Matrix:
 [[ 9783   166   415]
 [  142 12195   161]
 [  284   240 18859]]
Classification Report:
               precision    recall  f1-score   support

          -1       0.96      0.94      0.95     10364
           0       0.97      0.98      0.97     12498
           1       0.97      0.97      0.97     19383

    accuracy                           0.97     42245
   macro avg       0.97      0.96      0.96     42245
weighted avg       0.97      0.97      0.97     42245



In [ ]:
paeam_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

grid = GridSearchCV(log, param_grid=paeam_grid, cv=5, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


GridSearchCV(cv=5, estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10], 'penalty': ['l1', 'l2'],
                         'solver': ['liblinear']},
             verbose=1)

In [ ]:
print('best_parameters:', grid.best_params_)

best_model = grid.best_estimator_

y_pred = best_model.predict(x_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification Report:\n', classification_report(y_test, y_pred))

best_parameters: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Accuracy: 0.9845898922949462
Confusion Matrix:
 [[10126    67   171]
 [   66 12338    94]
 [  150   103 19130]]
Classification Report:
               precision    recall  f1-score   support

          -1       0.98      0.98      0.98     10364
           0       0.99      0.99      0.99     12498
           1       0.99      0.99      0.99     19383

    accuracy                           0.98     42245
   macro avg       0.98      0.98      0.98     42245
weighted avg       0.98      0.98      0.98     42245



# Word2vec Model

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 54.5 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

print("Imported Word2Vec from gensim.models.")

Imported Word2Vec from gensim.models.


In [ ]:
tokenized_text = [t.split() for t in df['text']]
print(f"Tokenized the first 5 entries of text data: {tokenized_text[:5]}")

Tokenized the first 5 entries of text data: [['That', 'game', 'hurt.'], ['>sexuality', 'shouldn’t', 'be', 'a', 'grouping', 'category', 'It', 'makes', 'you', 'different', 'from', 'othet', 'ppl', 'so', 'imo', 'it', 'fits', 'the', 'definition', 'of', '"grouping"'], ['You', 'do', 'right,', 'if', 'you', "don't", 'care', 'then', 'fuck', "'em!"], ['Man', 'I', 'love', 'reddit.'], ['[NAME]', 'was', 'nowhere', 'near', 'them,', 'he', 'was', 'by', 'the', 'Falcon.']]


In [ ]:
text = []
for i in range(len(df)):
  text.append(clean_text(df['text'][i]))

In [ ]:
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Re-load the DataFrame as it's no longer defined in the current scope.
df = pd.read_csv('/root/.cache/kagglehub/datasets/shivamb/go-emotions-google-emotions-dataset/versions/1/go_emotions_dataset.csv')

# Re-define the clean_text function, which is a dependency for processing the text data.
def clean_text(text_input):
  text_input = text_input.lower()
  text_input = re.sub(r'http\s+|www\S+|https\S+','',text_input,flags=re.MULTILINE)
  text_input = re.sub(r'\@\w+|\#','',text_input)
  text_input = text_input.translate(str.maketrans('','',string.punctuation))
  text_input = re.sub(r'\d+','',text_input)
  text_input = text_input.strip()

  stop_words = set(stopwords.words('english'))
  lemmatizer = WordNetLemmatizer()

  tokens = text_input.split()
  token = [lemmatizer.lemmatize(word) for word in tokens if not word in stop_words]
  text_input = ' '.join(token)

  return text_input

# Re-populate the 'text' list by applying the clean_text function to the DataFrame's text column.
text = []
for i in range(len(df)):
  text.append(clean_text(df['text'][i]))

# Tokenize the preprocessed text data as required by the subtask instructions.
tokenized_text = [t.split() for t in text]
print(f"Tokenized the first 5 entries of text data: {tokenized_text[:5]}")

Tokenized the first 5 entries of text data: [['game', 'hurt'], ['sexuality', 'shouldn’t', 'grouping', 'category', 'make', 'different', 'othet', 'ppl', 'imo', 'fit', 'definition', 'grouping'], ['right', 'dont', 'care', 'fuck', 'em'], ['man', 'love', 'reddit'], ['name', 'nowhere', 'near', 'falcon']]


In [ ]:
word2vec_model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=1, workers=4)
print("Word2Vec model trained successfully.")
print(f"Vocabulary size: {len(word2vec_model.wv)}")

Word2Vec model trained successfully.
Vocabulary size: 30185


In [ ]:
happy_words = ['joy', 'happy', 'delight', 'glad', 'euphoria', 'cheerful', 'ecstatic', 'thrilled']
sad_words = ['sad', 'grief', 'sorrow', 'unhappy', 'melancholy', 'depressed', 'despair', 'woe']
anger_words = ['angry', 'rage', 'fury', 'mad', 'frustration', 'irritation', 'resentment', 'wrath']

print("Emotion word lists defined.")

Emotion word lists defined.


In [ ]:
emotion_vectors = {
    'happy': {},
    'sad': {},
    'anger': {}
}

all_emotion_words = {
    'happy': happy_words,
    'sad': sad_words,
    'anger': anger_words
}

for emotion_type, words in all_emotion_words.items():
    print(f"Extracting vectors for {emotion_type} words:")
    for word in words:
        if word in word2vec_model.wv:
            emotion_vectors[emotion_type][word] = word2vec_model.wv[word]
            print(f"  - Extracted vector for '{word}'.")
        else:
            print(f"  - Word '{word}' not found in model's vocabulary.")

print("\nExtracted emotion vectors stored in 'emotion_vectors' dictionary.")
print(f"Number of happy word vectors extracted: {len(emotion_vectors['happy'])}")
print(f"Number of sad word vectors extracted: {len(emotion_vectors['sad'])}")
print(f"Number of anger word vectors extracted: {len(emotion_vectors['anger'])}")

Extracting vectors for happy words:
  - Extracted vector for 'joy'.
  - Extracted vector for 'happy'.
  - Extracted vector for 'delight'.
  - Extracted vector for 'glad'.
  - Word 'euphoria' not found in model's vocabulary.
  - Word 'cheerful' not found in model's vocabulary.
  - Extracted vector for 'ecstatic'.
  - Extracted vector for 'thrilled'.
Extracting vectors for sad words:
  - Extracted vector for 'sad'.
  - Extracted vector for 'grief'.
  - Extracted vector for 'sorrow'.
  - Extracted vector for 'unhappy'.
  - Word 'melancholy' not found in model's vocabulary.
  - Extracted vector for 'depressed'.
  - Extracted vector for 'despair'.
  - Extracted vector for 'woe'.
Extracting vectors for anger words:
  - Extracted vector for 'angry'.
  - Extracted vector for 'rage'.
  - Extracted vector for 'fury'.
  - Extracted vector for 'mad'.
  - Extracted vector for 'frustration'.
  - Word 'irritation' not found in model's vocabulary.
  - Extracted vector for 'resentment'.
  - Extracted v

In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. Prepare the data for t-SNE
all_vectors = []
all_labels = []

for emotion_type, vectors in emotion_vectors.items():
    for word, vector in vectors.items():
        all_vectors.append(vector)
        all_labels.append(emotion_type)

# 2. Convert the list of vectors and labels into NumPy arrays.
vectors_np = np.array(all_vectors)
labels_np = np.array(all_labels)

print(f"Shape of vectors_np: {vectors_np.shape}")
print(f"Shape of labels_np: {labels_np.shape}")
print("Data prepared for t-SNE.")

Shape of vectors_np: (20, 100)
Shape of labels_np: (20,)
Data prepared for t-SNE.


In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from gensim.models import Word2Vec # Import Word2Vec

# Re-define happy_words, sad_words, anger_words for self-containment
happy_words = ['joy', 'happy', 'delight', 'glad', 'euphoria', 'cheerful', 'ecstatic', 'thrilled']
sad_words = ['sad', 'grief', 'sorrow', 'unhappy', 'melancholy', 'depressed', 'despair', 'woe']
anger_words = ['angry', 'rage', 'fury', 'mad', 'frustration', 'irritation', 'resentment', 'wrath']

# Assuming tokenized_text is already available from previous cells, if not, it would need to be re-generated here.
# For robustness, we re-create tokenized_text using the clean_text function that was defined previously.
# Since df is in kernel state and clean_text function was defined, we can regenerate tokenized_text.

# Re-create word2vec_model (from cell 3b7648d5)
word2vec_model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=1, workers=4)
print("Word2Vec model re-trained successfully.")

# Re-create emotion_vectors (from cell b1544966)
emotion_vectors = {
    'happy': {},
    'sad': {},
    'anger': {}
}

all_emotion_words = {
    'happy': happy_words,
    'sad': sad_words,
    'anger': anger_words
}

for emotion_type, words in all_emotion_words.items():
    for word in words:
        if word in word2vec_model.wv:
            emotion_vectors[emotion_type][word] = word2vec_model.wv[word]

print("Extracted emotion vectors re-stored in 'emotion_vectors' dictionary.")

# 1. Prepare the data for t-SNE
all_vectors = []
all_labels = []

for emotion_type, vectors in emotion_vectors.items():
    for word, vector in vectors.items():
        all_vectors.append(vector)
        all_labels.append(emotion_type)

# 2. Convert the list of vectors and labels into NumPy arrays.
vectors_np = np.array(all_vectors)
labels_np = np.array(all_labels)

print(f"Shape of vectors_np: {vectors_np.shape}")
print(f"Shape of labels_np: {labels_np.shape}")
print("Data prepared for t-SNE.")

Word2Vec model re-trained successfully.
Extracted emotion vectors re-stored in 'emotion_vectors' dictionary.
Shape of vectors_np: (20, 100)
Shape of labels_np: (20,)
Data prepared for t-SNE.


In [ ]:
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=5)
vectors_3d = tsne_3d.fit_transform(vectors_np)

print(f"Shape of 3D t-SNE reduced vectors: {vectors_3d.shape}")
print("3D t-SNE dimensionality reduction completed.")

In [ ]:
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Create a numerical mapping for the labels if not already done (it is from the 2D plot, but for robustness)
label_to_numeric = {'happy': 0, 'sad': 1, 'anger': 2}
numeric_labels_3d = np.array([label_to_numeric[label] for label in labels_np])

# Define colors for each emotion type
emotion_colors = {'happy': 'green', 'sad': 'blue', 'anger': 'red'}
color_map_3d = [emotion_colors[label] for label in labels_np]

# Plot the 3D scatter plot
for label_idx, label_name in enumerate(label_to_numeric.keys()):
    indices = np.where(labels_np == label_name)
    ax.scatter(vectors_3d[indices, 0],
               vectors_3d[indices, 1],
               vectors_3d[indices, 2],
               color=emotion_colors[label_name],
               label=label_name,
               s=50)

ax.set_title('3D t-SNE Visualization of Emotion Word Embeddings')
ax.set_xlabel('t-SNE Component 1')
ax.set_ylabel('t-SNE Component 2')
ax.set_zlabel('t-SNE Component 3')
ax.legend(title='Emotion Type')
plt.show()

print("3D t-SNE visualization displayed.")

In [ ]:
print("\nPerforming analogy: love - like + hate")
analogy_result_2 = word2vec_model.wv.most_similar(positive=['love', 'hate'], negative=['like'], topn=5)

print("Top 5 most similar words to 'love - like + hate':")
for word, similarity in analogy_result_2:
    print(f"  {word}: {similarity:.4f}")

In [ ]:
print("Top 5 most similar words to 'happy':")
similar_to_happy = word2vec_model.wv.most_similar('happy', topn=5)
for word, similarity in similar_to_happy:
    print(f"  {word}: {similarity:.4f}")

In [ ]:
print("\nTop 5 most similar words to 'ecstatic':")
similar_to_ecstatic = word2vec_model.wv.most_similar('ecstatic', topn=5)
for word, similarity in similar_to_ecstatic:
    print(f"  {word}: {similarity:.4f}")

In [ ]:
similarity_happy_ecstatic = word2vec_model.wv.similarity('happy', 'ecstatic')
print(f"\nCosine similarity between 'happy' and 'ecstatic': {similarity_happy_ecstatic:.4f}")

In [ ]:
print("Top 5 most similar words to 'joy':")
similar_to_joy = word2vec_model.wv.most_similar('joy', topn=5)
for word, similarity in similar_to_joy:
    print(f"  {word}: {similarity:.4f}")

In [ ]:
print("\nTop 5 most similar words to 'joy':")
similar_to_joy = word2vec_model.wv.most_similar('joy', topn=5)
for word, similarity in similar_to_joy:
    print(f"  {word}: {similarity:.4f}")

In [ ]:
print("\nTop 5 most similar words to 'sadness':")
similar_to_sadness = word2vec_model.wv.most_similar('sadness', topn=5)
for word, similarity in similar_to_sadness:
    print(f"  {word}: {similarity:.4f}")

In [ ]:
similarity_joy_sadness = word2vec_model.wv.similarity('joy', 'sadness')
print(f"\nCosine similarity between 'joy' and 'sadness': {similarity_joy_sadness:.4f}")

In [ ]:
import numpy as np

# 1. Identify the emotion columns
emotion_columns = df.columns[df.columns.get_loc('admiration'):df.columns.get_loc('neutral')+1]

# 2. Initialize an empty dictionary for storing word embeddings per emotion
emotion_word_embeddings = {col: [] for col in emotion_columns}

# 3. Iterate through each row of the df DataFrame
for i in range(len(df)):
    # Access the corresponding cleaned and tokenized text
    current_text_tokens = tokenized_text[i]

    # 4. For each emotion column, check if the value in the current row is 1
    for emotion_col in emotion_columns:
        if df.loc[i, emotion_col] == 1:
            # For every word in the current tokenized_text:
            for word in current_text_tokens:
                # a. Check if the word exists in the word2vec_model.wv vocabulary
                if word in word2vec_model.wv:
                    # b. Retrieve its word embedding and append it to the list
                    emotion_word_embeddings[emotion_col].append(word2vec_model.wv[word])

# 5. Initialize an empty dictionary for storing the average embeddings (centroids)
emotion_centroids = {}

# 6. For each emotion, calculate the mean of all embeddings to get the centroid
for emotion_type, embeddings_list in emotion_word_embeddings.items():
    if embeddings_list:
        emotion_centroids[emotion_type] = np.mean(embeddings_list, axis=0)

# 8. Print the number of centroids calculated and the shape of one of the calculated centroids
print(f"Number of emotion centroids: {len(emotion_centroids)}")
if 'anger' in emotion_centroids:
    print(f"Shape of 'anger' centroid: {emotion_centroids['anger'].shape}")
else:
    print("Anger centroid not found (no 'anger' words with embeddings).")


In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# 1. Prepare the emotion centroids for t-SNE
centroid_vectors = []
centroid_labels = []

for emotion_type, centroid_vector in emotion_centroids.items():
    centroid_vectors.append(centroid_vector)
    centroid_labels.append(emotion_type)

centroid_vectors_np = np.array(centroid_vectors)
centroid_labels_np = np.array(centroid_labels)

print(f"Shape of centroid_vectors_np: {centroid_vectors_np.shape}")
print(f"Shape of centroid_labels_np: {centroid_labels_np.shape}")
print("Emotion centroids prepared for t-SNE.")

In [ ]:
tsne_centroids_2d = TSNE(n_components=2, random_state=42, perplexity=10) # Using perplexity=10 for 28 data points
centroids_2d_tsne = tsne_centroids_2d.fit_transform(centroid_vectors_np)

print(f"Shape of 2D t-SNE reduced centroids: {centroids_2d_tsne.shape}")
print("2D t-SNE dimensionality reduction for centroids completed.")

In [ ]:
plt.figure(figsize=(12, 10))
scatter = plt.scatter(centroids_2d_tsne[:, 0], centroids_2d_tsne[:, 1], c=centroid_labels_np, cmap='viridis')

# Create a legend
legend_handles = []
for label in np.unique(centroid_labels_np):
    idx = np.where(centroid_labels_np == label)[0][0]
    legend_handles.append(plt.Line2D([0], [0], marker='o', color='w', label=label,
                                      markerfacecolor=scatter.cmap(scatter.norm(idx)),
                                      markersize=10))
plt.legend(handles=legend_handles, title="Emotion Type", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title('2D t-SNE Visualization of Emotion Centroids')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.grid(True)
plt.tight_layout()
plt.show()

print("2D t-SNE visualization of emotion centroids displayed.")

In [ ]:
plt.figure(figsize=(12, 10))

# Map string labels to numerical values for coloring
unique_labels = np.unique(centroid_labels_np)
label_to_numeric_map = {label: i for i, label in enumerate(unique_labels)}
numeric_centroid_labels = np.array([label_to_numeric_map[label] for label in centroid_labels_np])

# Plot the scatter points using numerical labels for coloring
scatter = plt.scatter(centroids_2d_tsne[:, 0], centroids_2d_tsne[:, 1],
                      c=numeric_centroid_labels, cmap='viridis', s=100)

# Create a legend manually to show original string labels
legend_handles = []
for label_name, numeric_value in label_to_numeric_map.items():
    legend_handles.append(plt.Line2D([0], [0], marker='o', color='w', label=label_name,
                                      markerfacecolor=scatter.cmap(scatter.norm(numeric_value)),
                                      markersize=10))

plt.legend(handles=legend_handles, title="Emotion Type", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title('2D t-SNE Visualization of Emotion Centroids')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.grid(True)
plt.tight_layout()
plt.show()

print("2D t-SNE visualization of emotion centroids displayed.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# 4. Calculate the cosine similarity matrix between all pairs of emotion centroids
similarity_matrix = cosine_similarity(centroid_vectors_np)

# For better readability, convert to a DataFrame with emotion labels as index and columns
similarity_df = pd.DataFrame(similarity_matrix, index=centroid_labels_np, columns=centroid_labels_np)

print("Cosine Similarity Matrix calculated.")
print(similarity_df.head())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 5. Create a heatmap of the similarity_matrix
plt.figure(figsize=(14, 12))
sns.heatmap(similarity_df, annot=False, cmap='viridis', fmt=".2f", linewidths=.5)
plt.title('Cosine Similarity Matrix of Emotion Centroids')
plt.xlabel('Emotion')
plt.ylabel('Emotion')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("Heatmap of Cosine Similarity Matrix displayed.")